## Create numpy array of alzheimer patiant.

In [5]:
"""
Run local MAE-DFER inference on short windows from videos in ./data.

Expected project layout:
    project/
        inference.py
        data/                  # input videos (.mp4, .avi, .mov, .mkv, .webm)
        models/
            best.pth           # fine-tuned checkpoint from ElderReact training
        MAE-DFER/              # cloned https://github.com/sunlicai/MAE-DFER
        cache/                 # created automatically
        predictions.csv        # created automatically

Each video is split into non-overlapping 4-second clips. For each clip:
  1. Sample 32 evenly spaced frames with OpenCV.
  2. Detect a face in the middle, quarter, or three-quarter frame.
  3. Apply that same face crop to all frames (same strategy as training).
  4. Resize and cache as uint8 .npy.
  5. Select 16 deterministic frames (every other cached frame), normalize as in
     training, and run MAE-DFER.

Run from the project root:
    python inference.py
"""

from __future__ import annotations

import csv
import hashlib
import os
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import torch
from tqdm import tqdm

# ------------------------------ PATHS ---------------------------------
#PROJECT_DIR = Path(__file__).resolve().parent
DATA_DIR =   "data"
MODEL_PATH =  "models/best.pth"
REPO_DIR =   "MAE-DFER"
CACHE_DIR =   "cache/mae_dfer_s160_f32_face1"
OUTPUT_CSV =  "predictions.csv"

# ------------------------------ SETTINGS ------------------------------
CLIP_SECONDS = 4.0
N_CACHE_FRAMES = 32
NUM_FRAMES = 16
SIZE = 160
TUBELET = 2
BATCH_SIZE = 4
USE_FACE_CROP = True

MODEL_NAME = "vit_base_dim512_no_depth_patch16_160"
DEPTH = 16
ATTN_TYPE = "local_global"
LG_REGION_SIZE = [2, 5, 10]
LG_CLASSIFY_TOKEN_TYPE = "region"

EMOTIONS = ["anger", "disgust", "fear", "happiness", "sadness", "surprise"]
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# OpenCV configuration
cv2.ocl.setUseOpenCL(False)
cv2.setNumThreads(0)

# ------------------------------ DEVICE --------------------------------
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")


# --------------------------- FACE DETECTION ----------------------------
FACE_CASCADE = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)
if FACE_CASCADE.empty():
    raise RuntimeError("Could not load OpenCV's Haar face detector.")


def detect_face_box(rgb_frame: np.ndarray, margin: float = 0.3):
    """Return a square-ish expanded face crop (x0, y0, x1, y1), or None."""
    gray = cv2.cvtColor(rgb_frame, cv2.COLOR_RGB2GRAY)
    try:
        faces = FACE_CASCADE.detectMultiScale(
            gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40)
        )
    except cv2.error:
        return None

    if len(faces) == 0:
        return None

    x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
    height, width = rgb_frame.shape[:2]
    side = int(max(w, h) * (1.0 + margin))
    cx, cy = x + w // 2, y + h // 2

    x0 = max(0, cx - side // 2)
    y0 = max(0, cy - side // 2)
    x1 = min(width, x0 + side)
    y1 = min(height, y0 + side)

    # Keep the crop within the image while preserving its upper-left anchor.
    x0 = max(0, x1 - side)
    y0 = max(0, y1 - side)
    return x0, y0, x1, y1


# --------------------------- VIDEO SAMPLING ----------------------------
def sample_clip_frames(
    cap: cv2.VideoCapture,
    start_frame: int,
    end_frame: int,
    num_frames: int = N_CACHE_FRAMES,
):
    """
    Sample evenly spaced frames from [start_frame, end_frame).
    Returns RGB uint8 frames, or raises if no frames could be decoded.
    """
    if end_frame <= start_frame:
        raise ValueError("Empty clip interval.")

    # Midpoint sampling, matching the sampling convention in the training cache.
    indices = (
        (start_frame + (np.arange(num_frames) + 0.5)
         / num_frames * (end_frame - start_frame))
        .astype(np.int64)
    )
    indices = np.clip(indices, start_frame, end_frame - 1)

    frames = []
    for frame_idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, bgr = cap.read()
        if not ok:
            continue
        frames.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

    if len(frames) != num_frames:
        raise RuntimeError(
            f"Decoded {len(frames)}/{num_frames} frames for clip "
            f"starting at frame {start_frame}."
        )

    return np.stack(frames, axis=0)


def preprocess_clip(frames: np.ndarray) -> np.ndarray:
    """Apply one face crop to all frames, resize to 160x160, return uint8 K,H,W,3."""
    box = None

    if USE_FACE_CROP:
        # Same three candidate positions as the training preprocessing.
        for j in (len(frames) // 2, len(frames) // 4, 3 * len(frames) // 4):
            box = detect_face_box(frames[j])
            if box is not None:
                break

    if box is not None:
        x0, y0, x1, y1 = box
        frames = frames[:, y0:y1, x0:x1]

    resized = np.stack(
        [cv2.resize(frame, (SIZE, SIZE), interpolation=cv2.INTER_LINEAR)
         for frame in frames],
        axis=0,
    )
    return resized.astype(np.uint8)


# ------------------------------ CACHE ----------------------------------
def clip_cache_path(video_path: Path, clip_index: int) -> Path:
    key = hashlib.md5(str(video_path.resolve()).encode()).hexdigest()[:10]
    safe_stem = video_path.stem.replace(" ", "_")
    return CACHE_DIR / f"{safe_stem}_{key}_clip{clip_index:05d}.npy"


def load_or_create_clip(
    video_path: Path,
    cap: cv2.VideoCapture,
    clip_index: int,
    start_frame: int,
    end_frame: int,
) -> np.ndarray:
    path = clip_cache_path(video_path, clip_index)

    if path.is_file():
        try:
            arr = np.load(path)
            expected = (N_CACHE_FRAMES, SIZE, SIZE, 3)
            if arr.shape == expected and arr.dtype == np.uint8:
                return arr
        except (OSError, ValueError):
            pass

    frames = sample_clip_frames(cap, start_frame, end_frame)
    arr = preprocess_clip(frames)

    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.stem + ".tmp.npy")
    np.save(tmp_path, arr)
    os.replace(tmp_path, path)
    return arr


# ------------------------------- MODEL ---------------------------------
def build_model() -> torch.nn.Module:
    if not REPO_DIR.is_dir():
        raise FileNotFoundError(
            f"MAE-DFER repository not found at {REPO_DIR}.\n"
            "Clone it into the project folder with:\n"
            "git clone https://github.com/sunlicai/MAE-DFER.git MAE-DFER"
        )
    if not MODEL_PATH.is_file():
        raise FileNotFoundError(f"Fine-tuned checkpoint not found: {MODEL_PATH}")

    sys.path.insert(0, str(REPO_DIR))
    import modeling_finetune  # noqa: F401
    from timm.models import create_model

    model = create_model(
        MODEL_NAME,
        pretrained=False,
        num_classes=len(EMOTIONS),
        all_frames=NUM_FRAMES,
        tubelet_size=TUBELET,
        drop_path_rate=0.1,
        depth=DEPTH,
        attn_type=ATTN_TYPE,
        lg_region_size=LG_REGION_SIZE,
        lg_classify_token_type=LG_CLASSIFY_TOKEN_TYPE,
    )

    checkpoint = torch.load(
        MODEL_PATH, map_location="cpu", weights_only=False
    )
    state_dict = checkpoint.get("model", checkpoint)
    model.load_state_dict(state_dict, strict=True)
    model.to(DEVICE)
    model.eval()

    epoch = checkpoint.get("epoch", "unknown") if isinstance(checkpoint, dict) else "unknown"
    score = checkpoint.get("score", "unknown") if isinstance(checkpoint, dict) else "unknown"
    print(f"Loaded fine-tuned checkpoint: {MODEL_PATH}")
    print(f"Checkpoint epoch: {epoch} | validation score: {score}")
    return model


# ------------------------- TENSOR PREPARATION --------------------------
def cached_clip_to_tensor(arr: np.ndarray) -> torch.Tensor:
    """
    Match validation preprocessing:
      cached 32 frames -> take indices 0,2,...,30 -> normalize -> C,T,H,W.
    """
    selected = arr[:: N_CACHE_FRAMES // NUM_FRAMES]
    if selected.shape[0] != NUM_FRAMES:
        raise ValueError(f"Expected {NUM_FRAMES} selected frames, got {selected.shape[0]}")

    x = (selected.astype(np.float32) / 255.0 - MEAN) / STD
    x = np.ascontiguousarray(x.transpose(3, 0, 1, 2))
    return torch.from_numpy(x)


# ------------------------------- INFERENCE -----------------------------
@torch.inference_mode()
def predict_video(model: torch.nn.Module, video_path: Path, writer):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"WARNING: could not open {video_path}; skipping.")
        return 0

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if not np.isfinite(fps) or fps <= 0 or total_frames <= 0:
        cap.release()
        print(f"WARNING: invalid video metadata for {video_path}; skipping.")
        return 0

    duration = total_frames / fps
    frames_per_clip = max(1, int(round(CLIP_SECONDS * fps)))
    n_clips = int(np.ceil(total_frames / frames_per_clip))

    print(
        f"\n{video_path.name}: {duration / 60:.2f} min, "
        f"{fps:.2f} FPS, {n_clips} clips of up to {CLIP_SECONDS:g}s"
    )

    pending_tensors = []
    pending_rows = []
    written = 0

    def run_batch():
        nonlocal written
        if not pending_tensors:
            return

        batch = torch.stack(pending_tensors).to(DEVICE)
        if DEVICE.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(batch)
        else:
            logits = model(batch)

        probabilities = torch.sigmoid(logits.float()).cpu().numpy()

        for metadata, probs in zip(pending_rows, probabilities):
            video_name, clip_idx, start_s, end_s = metadata
            writer.writerow(
                [video_name, clip_idx, f"{start_s:.3f}", f"{end_s:.3f}"]
                + [f"{float(p):.6f}" for p in probs]
            )
            written += 1

        pending_tensors.clear()
        pending_rows.clear()

    start_time = time.time()

    try:
        for clip_idx in tqdm(range(n_clips), desc=video_path.name, leave=False):
            start_frame = clip_idx * frames_per_clip
            end_frame = min(total_frames, start_frame + frames_per_clip)

            try:
                arr = load_or_create_clip(
                    video_path, cap, clip_idx, start_frame, end_frame
                )
                tensor = cached_clip_to_tensor(arr)
            except Exception as exc:
                print(f"\nWARNING: {video_path.name}, clip {clip_idx}: {exc}")
                continue

            pending_tensors.append(tensor)
            pending_rows.append(
                (
                    video_path.name,
                    clip_idx,
                    start_frame / fps,
                    min(end_frame / fps, duration),
                )
            )

            if len(pending_tensors) >= BATCH_SIZE:
                run_batch()

        run_batch()
    finally:
        cap.release()

    elapsed = time.time() - start_time
    print(f"Finished {video_path.name}: {written}/{n_clips} clips in {elapsed:.1f}s")
    return written


def main():
    if not DATA_DIR.is_dir():
        raise FileNotFoundError(f"Input folder not found: {DATA_DIR}")

    video_paths = sorted(
        p for p in DATA_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in VIDEO_EXTS
    )
    if not video_paths:
        raise FileNotFoundError(
            f"No supported video files found in {DATA_DIR}"
        )

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    model = build_model()

    total_written = 0
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

    with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(
            ["video", "clip_index", "start_seconds", "end_seconds"]
            + [f"prob_{emotion}" for emotion in EMOTIONS]
        )

        for video_path in video_paths:
            total_written += predict_video(model, video_path, writer)
            f.flush()

    print(f"\nSaved {total_written} clip predictions to: {OUTPUT_CSV}")
    print("Each emotion probability is an independent sigmoid probability; "
          "the six values do not sum to 1.")


if __name__ == "__main__":
    main()


Using device: mps


[ERROR:0@1104.794] global persistence.cpp:584 open Can't open file: '/Users/isakhamrin/Programmering/year5/project_course/Scientific-computing-project---Emotion-detection/venv/lib/python3.11/site-packages/cv2/data/haarcascade_frontalface_default.xml' in read mode


RuntimeError: Could not load OpenCV's Haar face detector.

In [6]:
import cv2
import os

print("OpenCV version:", cv2.__version__)
print("OpenCV data path:", cv2.data.haarcascades)
print("Directory exists:", os.path.exists(cv2.data.haarcascades))
print("Cascade files:", os.listdir(cv2.data.haarcascades)
      if os.path.exists(cv2.data.haarcascades) else "Directory missing")

OpenCV version: 5.0.0
OpenCV data path: /Users/isakhamrin/Programmering/year5/project_course/Scientific-computing-project---Emotion-detection/venv/lib/python3.11/site-packages/cv2/data/
Directory exists: True
Cascade files: ['__init__.py', '__pycache__']
